# Week 7: 线性规划

## 学习目标

1. 理解线性规划的基本概念
2. 学会使用 PuLP 库建模和求解
3. 掌握典型的线性规划应用
4. 理解对偶和灵敏度分析

## 1. 线性规划基础

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pulp import *

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("线性规划工具已加载")

### 1.1 标准形式

线性规划问题的标准形式：

**目标函数**：最大化或最小化 c^T x

**约束条件**：
- Ax ≤ b
- x ≥ 0

### 1.2 图解法示例

In [ ]:
# 简单示例：
# 最大化 z = 3x + 2y
# 约束：
#   x + y ≤ 4
#   2x + y ≤ 5
#   x, y ≥ 0

fig, ax = plt.subplots(figsize=(10, 8))

# 绘制约束线
x = np.linspace(0, 5, 100)

# x + y ≤ 4 => y ≤ 4 - x
y1 = 4 - x
# 2x + y ≤ 5 => y ≤ 5 - 2x
y2 = 5 - 2*x

ax.plot(x, y1, 'r-', label='x + y = 4')
ax.plot(x, y2, 'b-', label='2x + y = 5')
ax.axhline(0, color='black', lw=0.5)
ax.axvline(0, color='black', lw=0.5)

# 可行域
vertices = [(0, 0), (0, 4), (1, 3), (2.5, 0)]
from matplotlib.patches import Polygon
polygon = Polygon(vertices, alpha=0.3, color='green')
ax.add_patch(polygon)

# 标记顶点
for vx, vy in vertices:
    ax.plot(vx, vy, 'ko', markersize=8)
    z = 3*vx + 2*vy
    ax.annotate(f'({vx:.1f}, {vy:.1f})\nz={z:.1f}', 
                xy=(vx, vy), xytext=(vx+0.2, vy+0.3))

ax.set_xlim(-0.5, 5)
ax.set_ylim(-0.5, 5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('线性规划图解法')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("最优解在顶点 (1, 3)，最大值 z = 9")

## 2. 使用 PuLP 建模

### 2.1 基本示例

In [ ]:
# 创建问题
prob = LpProblem("Simple_Example", LpMaximize)

# 定义变量（非负）
x = LpVariable("x", lowBound=0)
y = LpVariable("y", lowBound=0)

# 目标函数
prob += 3*x + 2*y, "Objective"

# 约束条件
prob += x + y <= 4, "Constraint_1"
prob += 2*x + y <= 5, "Constraint_2"

# 求解
prob.solve()

# 输出结果
print("状态:", LpStatus[prob.status])
print("最优解:")
for v in prob.variables():
    print(f"  {v.name} = {v.varValue}")
print(f"目标函数值 = {value(prob.objective)}")

## 3. 生产计划问题

In [ ]:
# 问题：工厂生产两种产品 A 和 B
# 产品 A 利润 300 元/件，产品 B 利润 500 元/件
# 约束：
#   机器时间：A 需要 2 小时，B 需要 3 小时，总可用 100 小时
#   原材料：A 需要 4 kg，B 需要 2 kg，总可用 120 kg
#   市场需求：B 最多生产 30 件

prob = LpProblem("Production_Planning", LpMaximize)

# 变量：生产数量
A = LpVariable("Product_A", lowBound=0, cat='Integer')
B = LpVariable("Product_B", lowBound=0, cat='Integer')

# 目标：最大化利润
prob += 300*A + 500*B, "Total_Profit"

# 约束
prob += 2*A + 3*B <= 100, "Machine_Time"
prob += 4*A + 2*B <= 120, "Material"
prob += B <= 30, "Market_Demand"

# 求解
prob.solve()

print("生产计划优化结果")
print("=" * 40)
print("状态:", LpStatus[prob.status])
for v in prob.variables():
    print(f"{v.name} = {v.varValue} 件")
print(f"\n最大利润 = {value(prob.objective)} 元")

In [ ]:
# 可视化可行域和最优解
fig, ax = plt.subplots(figsize=(10, 8))

A_range = np.linspace(0, 40, 100)

# 约束线
B1 = (100 - 2*A_range) / 3  # 机器时间
B2 = (120 - 4*A_range) / 2  # 原材料
B3 = 30 * np.ones_like(A_range)  # 市场需求

ax.plot(A_range, B1, 'r-', label='机器时间')
ax.plot(A_range, B2, 'b-', label='原材料')
ax.plot(A_range, B3, 'g-', label='市场需求')

# 最优解
A_opt = A.varValue
B_opt = B.varValue
ax.plot(A_opt, B_opt, 'ko', markersize=12, label=f'最优解 ({A_opt:.0f}, {B_opt:.0f})')

ax.set_xlim(0, 40)
ax.set_ylim(0, 40)
ax.set_xlabel('产品 A 数量')
ax.set_ylabel('产品 B 数量')
ax.set_title('生产计划优化')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 运输问题

In [ ]:
# 运输问题：从 3 个工厂运往 4 个仓库
# 目标：最小化运输成本

# 工厂供应量
supply = {'F1': 100, 'F2': 150, 'F3': 200}

# 仓库需求量
demand = {'W1': 80, 'W2': 120, 'W3': 150, 'W4': 100}

# 运输成本（元/件）
costs = {
    ('F1', 'W1'): 3, ('F1', 'W2'): 5, ('F1', 'W3'): 7, ('F1', 'W4'): 6,
    ('F2', 'W1'): 2, ('F2', 'W2'): 5, ('F2', 'W3'): 8, ('F2', 'W4'): 4,
    ('F3', 'W1'): 3, ('F3', 'W2'): 6, ('F3', 'W3'): 9, ('F3', 'W4'): 2
}

# 创建问题
prob = LpProblem("Transportation", LpMinimize)

# 变量：运输量
routes = [(f, w) for f in supply for w in demand]
transport = LpVariable.dicts("route", routes, lowBound=0, cat='Integer')

# 目标函数
prob += lpSum([costs[r] * transport[r] for r in routes]), "Total_Cost"

# 供应约束
for f in supply:
    prob += lpSum([transport[(f, w)] for w in demand]) <= supply[f], f"Supply_{f}"

# 需求约束
for w in demand:
    prob += lpSum([transport[(f, w)] for f in supply]) >= demand[w], f"Demand_{w}"

# 求解
prob.solve()

print("运输问题优化结果")
print("=" * 40)
print("状态:", LpStatus[prob.status])
print(f"\n最小运输成本 = {value(prob.objective)} 元\n")

print("运输方案:")
for r in routes:
    if transport[r].varValue > 0:
        print(f"  {r[0]} -> {r[1]}: {transport[r].varValue} 件")

In [ ]:
# 可视化运输方案
result_matrix = np.zeros((len(supply), len(demand)))
cost_matrix = np.zeros((len(supply), len(demand)))

factories = list(supply.keys())
warehouses = list(demand.keys())

for i, f in enumerate(factories):
    for j, w in enumerate(warehouses):
        result_matrix[i, j] = transport[(f, w)].varValue
        cost_matrix[i, j] = costs[(f, w)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 运输量热力图
import seaborn as sns
sns.heatmap(result_matrix, annot=True, fmt='.0f', cmap='Blues', 
            xticklabels=warehouses, yticklabels=factories, ax=axes[0])
axes[0].set_xlabel('仓库')
axes[0].set_ylabel('工厂')
axes[0].set_title('运输量')

# 成本热力图
sns.heatmap(cost_matrix, annot=True, fmt='.0f', cmap='Reds',
            xticklabels=warehouses, yticklabels=factories, ax=axes[1])
axes[1].set_xlabel('仓库')
axes[1].set_ylabel('工厂')
axes[1].set_title('单位成本')

plt.tight_layout()
plt.show()

## 5. 单车调度问题

In [ ]:
# 问题：平衡各站点单车数量
# 目标：最小化总移动成本

# 站点当前数量和目标数量
stations = ['S1', 'S2', 'S3', 'S4']
current = {'S1': 30, 'S2': 10, 'S3': 50, 'S4': 20}
target = {'S1': 25, 'S2': 25, 'S3': 30, 'S4': 30}

# 站点间移动成本（元/辆）
move_cost = {
    ('S1', 'S2'): 2, ('S1', 'S3'): 3, ('S1', 'S4'): 4,
    ('S2', 'S1'): 2, ('S2', 'S3'): 2, ('S2', 'S4'): 3,
    ('S3', 'S1'): 3, ('S3', 'S2'): 2, ('S3', 'S4'): 2,
    ('S4', 'S1'): 4, ('S4', 'S2'): 3, ('S4', 'S3'): 2
}

# 计算各站点盈亏
surplus = {s: current[s] - target[s] for s in stations}

print("站点单车盈亏情况")
print("=" * 40)
for s in stations:
    status = "盈余" if surplus[s] > 0 else "亏损" if surplus[s] < 0 else "平衡"
    print(f"{s}: 当前 {current[s]}, 目标 {target[s]}, {status} {abs(surplus[s])} 辆")

In [ ]:
# 建模求解
prob = LpProblem("Bike_Rebalancing", LpMinimize)

# 变量：移动量
routes = [(i, j) for i in stations for j in stations if i != j]
move = LpVariable.dicts("move", routes, lowBound=0, cat='Integer')

# 目标：最小化成本
prob += lpSum([move_cost[r] * move[r] for r in routes]), "Total_Cost"

# 平衡约束
for s in stations:
    outflow = lpSum([move[(s, j)] for j in stations if s != j])
    inflow = lpSum([move[(i, s)] for i in stations if s != i])
    prob += outflow - inflow == surplus[s], f"Balance_{s}"

# 求解
prob.solve()

print("\n单车调度方案")
print("=" * 40)
print("状态:", LpStatus[prob.status])
print(f"\n最小移动成本 = {value(prob.objective)} 元\n")

print("调度方案:")
for r in routes:
    if move[r].varValue > 0:
        print(f"  {r[0]} -> {r[1]}: {move[r].varValue} 辆")

## 6. Research Thinking

### 问题 1：整数规划 vs 线性规划

什么时候需要整数规划？

**回答：**

- **需要整数解**：产品数量、人员分配
- **可以放松**：生产量很大时，小数误差可忽略
- **计算复杂度**：整数规划比线性规划难得多

### 问题 2：灵敏度分析

参数变化对最优解的影响？

**回答：**

- **影子价格**：约束资源增加一单位带来的目标改进
- **允许范围**：参数变化不改变最优基的范围
- **实际意义**：判断哪些资源最宝贵

### 问题 3：模型局限性

线性规划的假设可能不成立？

**回答：**

1. **线性假设**：实际可能存在规模经济
2. **确定性假设**：参数可能不确定
3. **静态假设**：没有考虑时间维度
4. **单一目标**：实际可能有多目标

## 7. 练习

### 练习 1
求解以下线性规划问题：
- 最大化 z = 2x + 3y
- 约束：x + 2y ≤ 8, 3x + y ≤ 9, x, y ≥ 0

In [ ]:
# 你的代码


### 练习 2
扩展运输问题，考虑运输能力限制。

In [ ]:
# 你的代码


### 练习 3
设计一个投资组合优化问题。

In [ ]:
# 你的代码


## 8. 总结

### 本周学习要点

1. **线性规划基础**：目标函数、约束条件、可行域
2. **PuLP 建模**：问题定义、变量、约束、求解
3. **典型应用**：生产计划、运输问题、资源分配
4. **对偶分析**：影子价格、灵敏度

### 关键洞察

- 线性规划是优化的基础工具
- 建模比求解更重要
- 理解约束的本质有助于改进系统